# RAG con LangChain y citas de fuentes

Cadena RAG con identificadores de documentos y generacion mediante Qwen2.5-0.5B-Instruct.

In [1]:
%%capture
!pip -q install "langchain>=1.0.0" "langchain-community>=0.4.0" "faiss-cpu>=1.8.0" "transformers>=4.45.0" "torch"

In [2]:
import os
from typing import List

import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
hf_logging.set_verbosity_error()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()


class TransformerEmbeddings(Embeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            output = model(**inputs, output_hidden_states=True)
        hidden = output.hidden_states[-1][0]
        mask = inputs["attention_mask"][0].unsqueeze(-1)
        vector = (hidden * mask).sum(dim=0) / mask.sum()
        return vector.detach().cpu().tolist()


def hf_generate(prompt_value):
    prompt = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    messages = [
        {"role": "system", "content": "Eres un asistente técnico. Responde SIEMPRE en español. Usa solo el contexto. RAG significa Retrieval-Augmented Generation; no inventes otro significado."},
        {"role": "user", "content": prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


print("Modelo y embeddings reales cargados:", MODEL_NAME)

/Users/marcechris/Documents/Codex/2026-06-22/a/work/notebook_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/vd/swgm5nfx6tv28vjv7wl7bgxw0000gn/T/ipykernel_14562/2161163042.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 14861.24it/s]

Modelo y embeddings reales cargados: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
base_conocimiento = [
    Document(page_content="Un sistema RAG recupera fragmentos relevantes antes de generar una respuesta.", metadata={"id": "DOC-01", "titulo": "Definicion RAG"}),
    Document(page_content="RAG puede reducir respuestas inventadas al conectar el modelo con fuentes especificas.", metadata={"id": "DOC-02", "titulo": "Ventajas"}),
    Document(page_content="Un riesgo de RAG es recuperar documentos irrelevantes o incompletos.", metadata={"id": "DOC-03", "titulo": "Riesgos"}),
]

vectorstore = FAISS.from_documents(base_conocimiento, TransformerEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

pregunta = "Responde SOLO en español. Explica por qué Retrieval-Augmented Generation (RAG) es útil en un pipeline de IA y menciona el riesgo de recuperar documentos irrelevantes o incompletos."
recuperados = retriever.invoke(pregunta)

for doc in recuperados:
    print(f"[{doc.metadata['id']}] {doc.page_content}")

[DOC-01] Un sistema RAG recupera fragmentos relevantes antes de generar una respuesta.
[DOC-02] RAG puede reducir respuestas inventadas al conectar el modelo con fuentes especificas.
[DOC-03] Un riesgo de RAG es recuperar documentos irrelevantes o incompletos.


In [4]:
def serializar_con_citas(docs):
    return "\n".join(f"[{doc.metadata['id']}] {doc.page_content}" for doc in docs)


prompt = ChatPromptTemplate.from_template(
    """Contesta SOLO en español usando el contexto y conserva citas como [DOC-01]. RAG significa Retrieval-Augmented Generation. No uses inglés ni inventes otro significado.

Contexto:
{context}

Pregunta:
{question}

Respuesta completa en dos frases:"""
)

cadena_con_citas = (
    {"context": retriever | RunnableLambda(serializar_con_citas), "question": RunnablePassthrough()}
    | prompt
    | RunnableLambda(hf_generate)
    | StrOutputParser()
)

respuesta = cadena_con_citas.invoke(pregunta)
print(respuesta)

Retrieval-Augmented Generation (RAG) es útil en un pipeline de IA porque permite recuperar información relevante antes de generar una respuesta, lo que reduce la probabilidad de errores y mejora la precisión del resultado. Sin embargo, este proceso también tiene riesgos, ya que puede recuperar documentos irrelevantes o incompletos, lo cual puede afectar la calidad final de las respuestas generadas.
